<a href="https://colab.research.google.com/github/miso-20/ESSA/blob/main/ESAA_YB_WEEK_16_1-project_review2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **프로젝트 수상작 리뷰**
영화 관객수 예측 경진대회

https://dacon.io/competitions/open/235536/codeshare/2721?page=1&dtype=recent


## 주제
영화 관객 수(box_off_num)를 예측하는 머신러닝 회귀 모델링


## 데이터

movies_train.csv / movies_test.csv

- title : 영화의 제목
- distributor : 배급사
- genre : 장르
- release_time : 개봉일
- time : 상영시간(분)
- screening_rat : 상영등급
- director : 감독이름
- dir_prev_bfnum : 해당 감독이 이 영화를 만들기 전 제작에 참여한 영화에서의 평균 관객수(단 관객수가 알려지지 않은 영화 제외)
- dir_prev_num : 해당 감독이 이 영화를 만들기 전 제작에 참여한 영화의 개수(단 관객수가 알려지지 않은 영화 제외)
- num_staff : 스텝수
- num_actor : 주연배우수
- box_off_num : 관객수


## 코드 흐름

### 1. 데이터 전처리 (Preprocessing)

- 배급사(distributor) 정제: '(주)' 및 특수문자 제거 후 비슷한 계열사(CJ, 쇼박스 등) 통폐합

- 랭크 인코딩 (Rank Encoding):

  1) 장르(genre): 장르별 관객 수 평균값을 기준으로 1~12위까지 순위화하여 변환

  2) 배급사(distributor): 배급사별 관객 수 중앙값을 기준으로 순위화하여 변환 (num_rank)

- 변수 변환: screening_rat은 더미 변수(One-Hot Encoding)로 변환하고, 편차가 큰 num_actor(배우 수)와 타겟 변수인 관객 수(box_off_num)는 로그 변환(np.log1p) 수행



### 2. 교차 검증 및 모델 학습

- KFold(n_splits=10)를 활용하여 10-Fold 교차 검증 수행

- 총 6가지 트리 기반 회귀 모델 사용: GBM, NGBRegressor, LGBM, XGBoost, CatBoost, RandomForest

### 3. 최종 예측 (Ensemble)

- 6개 모델에서 도출된 예측값을 평균 내는 블렌딩(Blending) 방식으로 최종 관객 수 산출





**주요 코드**

In [ ]:
# 타겟 변수 및 편차가 큰 독립 변수 로그 변환
y = np.log1p(train.box_off_num)
X['num_actor'] = np.log1p(X['num_actor'])

# 범주형 변수(상영 등급) 원핫 인코딩
X = pd.get_dummies(columns = ['screening_rat'], data = X)

# 10-Fold 교차검증 세팅 및 6개 모델 블렌딩을 통한 최종 예측
kf = KFold(n_splits =  10, shuffle = True, random_state = 42)

# 각 모델별 학습 및 예측 수행
submission['box_off_num'] = (xgb_pred + cat_pred + lgb_pred + rf_pred + gb_pred + ngb_pred) / 6

## 새롭게 알게 된 내용 / 어려운 내용 / 배울 점

- 타겟 변수 기반의 랭크 인코딩 기법 : 범주형 변수(장르, 배급사)를 단순히 라벨 인코딩하거나 원핫 인코딩하지 않고, 타겟 변수인 '관객 수'의 평균이나 중앙값을 기준으로 순위를 매겨 인코딩하는 방식이 모델 성능 향상에 매우 직관적이고 효과적이라는 것을 배웠다.

- 로그 변환의 중요성 : 관객 수나 배우 수처럼 값의 편차가 크고 한쪽으로 치우쳐진 데이터에 np.log1p를 적용하여 정규분포에 가깝게 만들어주는 전처리가 회귀 예측에서 필수적임을 알게 되었다. 최종 예측 시에는 다시 np.expm1로 복원하는 디테일도 확인할 수 있었다.

- 다양한 모델 블렌딩 앙상블 : 단일 모델에 의존하지 않고 GBM, XGB, LGBM 등 6가지의 강력한 부스팅 및 배깅 모델을 10-Fold 교차검증으로 학습시킨 뒤, 결과값을 1/N로 블렌딩하여 과적합을 방지하고 예측의 안정성을 극대화하는 전략을 배웠다.